# AnimeLoom — RunPod Text-to-Anime

Generate studio-quality anime video from text on RunPod GPUs.

| Stage | What | Purpose |
|-------|------|---------|
| **Story Decomposer** | Gemini Flash (free) or rule-based | Text → shot script |
| **SDXL + LoRA** | Keyframe generation | Character identity via trained LoRA |
| **AnimateDiff T2V** | Text-to-video + IP-Adapter | Real motion with character conditioning |
| **Motion Trim** | Remove static tails | Cleaner clip endings |
| **RIFE** | Temporal upscale 8→24fps | Smooth animation |
| **Real-ESRGAN** | Spatial sharpening | Crisp anime lines |
| **Face Restore** | Anime face enhancement | Sharper faces |
| **Color Grade** | Anime palette | Vibrant studio colors |
| **Assembly** | Cross-dissolve stitch | Smooth scene transitions |

In [ ]:
#@title 1. Setup Environment
#@markdown Run this cell once after pod starts. Installs pinned dependencies.

import subprocess, sys, os
from pathlib import Path

# Clone repo if needed
os.chdir("/workspace")
if not Path("/workspace/AnimeLoom").exists():
    subprocess.run(["git", "clone", "https://github.com/JoelJohnsonThomas/AnimeLoom.git"], check=True)
os.chdir("/workspace/AnimeLoom")
sys.path.insert(0, "/workspace/AnimeLoom")

# Memory optimization — reduce fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Upgrade torch to 2.5+ (required for diffusers 0.36 Wan pipeline — enable_gqa support)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu124",
], check=False)

# Pinned deps — compatible with torch 2.5.1+cu124
# diffusers 0.36.0 needed for Wan2.1 I2V pipeline support
deps = [
    "diffusers==0.36.0", "transformers==4.44.2", "accelerate==0.33.0", "numpy<2.0",
    "safetensors", "peft", "sentencepiece", "protobuf", "ftfy",
    "opencv-python-headless", "pillow", "scikit-image", "scikit-learn",
    "controlnet-aux", "einops", "omegaconf",
    "realesrgan", "basicsr==1.4.2", "gfpgan", "facexlib",
    "google-genai", "google-generativeai",
    "huggingface_hub",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps)

# Fix total_mem → total_memory for torch 2.5
wan_path = Path("agents/animator/wan_wrapper.py")
if wan_path.exists():
    code = wan_path.read_text()
    if ".total_mem" in code and ".total_memory" not in code:
        wan_path.write_text(code.replace(".total_mem", ".total_memory"))
        print("Fixed total_mem → total_memory")

# Set warehouse
os.environ["AI_CACHE_ROOT"] = "/workspace/warehouse"
for d in ["models", "lora", "outputs", "checkpoints", "references"]:
    Path(f"/workspace/warehouse/{d}").mkdir(parents=True, exist_ok=True)

# torchvision compatibility shim for basicsr
import torchvision, types
if not hasattr(torchvision.transforms, 'functional_tensor'):
    import torchvision.transforms.functional as F
    m = types.ModuleType('torchvision.transforms.functional_tensor')
    for a in ['rgb_to_grayscale', 'normalize', 'resize', 'pad']:
        if hasattr(F, a): setattr(m, a, getattr(F, a))
    sys.modules['torchvision.transforms.functional_tensor'] = m

import torch
print(f"torch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB)")
print(f"Warehouse: {os.environ['AI_CACHE_ROOT']}")
print(f"CUDA alloc config: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF', 'default')}")

import diffusers
print(f"diffusers {diffusers.__version__}")
print("Setup complete!")

In [ ]:
#@title 2. Download Character LoRA from HuggingFace
#@markdown Downloads pre-trained character LoRA. Skip if you'll train your own.

CHARACTER_REPO = "joelthomas77/animeloom-sakura-haruno"  #@param {type:"string"}
CHARACTER_NAME = "sakura_haruno"  #@param {type:"string"}

from huggingface_hub import snapshot_download
import shutil
from pathlib import Path

WAREHOUSE = Path("/workspace/warehouse")
hf_dir = WAREHOUSE / "lora" / "_hf_download"

print(f"Downloading {CHARACTER_REPO}...")
snapshot_download(CHARACTER_REPO, local_dir=str(hf_dir))

# Place SDXL and SD 1.5 LoRAs in correct directories
for src, suffix in [("sdxl", ""), ("sd15", "_sd15")]:
    src_dir = hf_dir / src
    if src_dir.exists():
        dst_dir = WAREHOUSE / "lora" / f"{CHARACTER_NAME}{suffix}"
        dst_dir.mkdir(parents=True, exist_ok=True)
        for f in src_dir.iterdir():
            shutil.copy2(f, dst_dir / f.name)
        print(f"  {src} → {dst_dir} ({sum(1 for _ in dst_dir.iterdir())} files)")

# Verify
sdxl_ok = (WAREHOUSE / "lora" / CHARACTER_NAME / "adapter_model.safetensors").exists()
sd15_ok = (WAREHOUSE / "lora" / f"{CHARACTER_NAME}_sd15" / "adapter_model.safetensors").exists()
print(f"\nSDXL LoRA: {'OK' if sdxl_ok else 'MISSING'}")
print(f"SD 1.5 LoRA (AnimateDiff): {'OK' if sd15_ok else 'MISSING'}")

In [ ]:
#@title 3. Text-to-Anime Generation (Wan2.1 I2V)
#@markdown Type a story and get anime video with character consistency.
#@markdown Uses SDXL+LoRA keyframes → Wan2.1 Image-to-Video for studio anime quality.

#@markdown ---
#@markdown **Your Story**
STORY_TEXT = "Sakura Haruno walks through a cherry blossom forest at sunset. Petals fall around her as she stops at a wooden bridge and looks at the river below. The wind gently moves her pink hair."  #@param {type:"string"}

#@markdown ---
#@markdown **Character** (must match LoRA from Step 2, or leave empty)
CHARACTER_NAME = "sakura_haruno"  #@param {type:"string"}

#@markdown ---
#@markdown **Gemini API Key** (optional — get free at aistudio.google.com/apikey)
GEMINI_API_KEY = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Video Settings**
NUM_FRAMES = 33  #@param {type:"slider", min:17, max:81, step:8}
WAN_STEPS = 30  #@param {type:"slider", min:15, max:50, step:5}
WAN_GUIDANCE = 5.0  #@param {type:"slider", min:1.0, max:10.0, step:0.5}
FPS = 16  #@param {type:"slider", min:8, max:24, step:4}
TARGET_FPS = 24  #@param {type:"slider", min:8, max:30, step:4}

#@markdown ---
#@markdown **Post-Processing**
FACE_RESTORE = True  #@param {type:"boolean"}
SPATIAL_UPSCALE = True  #@param {type:"boolean"}
COLOR_GRADE = True  #@param {type:"boolean"}
COLOR_PALETTE = "warm"  #@param ["warm", "cool", "vibrant", "muted"]

#@markdown ---
#@markdown **SDXL Keyframe Settings**
IMAGE_WIDTH = 768  #@param {type:"slider", min:512, max:1024, step:128}
IMAGE_HEIGHT = 1152  #@param {type:"slider", min:512, max:1152, step:128}
SDXL_STEPS = 25  #@param {type:"slider", min:10, max:50, step:5}
SDXL_GUIDANCE = 7.5  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
LORA_SCALE = 1.0  #@param {type:"slider", min:0.5, max:2.0, step:0.1}
NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality, ugly, disfigured, bad anatomy, bad hands, 3d render, cgi, photorealistic, live action, multiple views, split screen"  #@param {type:"string"}

import os, gc, sys, time, torch, json, re
import numpy as np
from pathlib import Path
from PIL import Image
import cv2

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# ================================================================
# Auto-detect GPU VRAM
# ================================================================
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if VRAM_GB >= 40:
    WAN_W, WAN_H = 480, 832
    OFFLOAD_MODE = "model"
    print(f"GPU: {VRAM_GB:.0f}GB — full resolution {WAN_W}x{WAN_H}, model CPU offload (fast)")
elif VRAM_GB >= 20:
    WAN_W, WAN_H = 480, 640
    OFFLOAD_MODE = "sequential"
    print(f"GPU: {VRAM_GB:.0f}GB — reduced resolution {WAN_W}x{WAN_H}, sequential CPU offload")
else:
    WAN_W, WAN_H = 480, 480
    OFFLOAD_MODE = "sequential"
    print(f"GPU: {VRAM_GB:.0f}GB — minimal resolution {WAN_W}x{WAN_H}")

# ================================================================
# Helpers
# ================================================================

def wan_frame_to_pil(frame):
    """Convert Wan output frame to PIL — handles float32, float16, tensors."""
    if isinstance(frame, Image.Image):
        return frame
    if hasattr(frame, 'cpu'):
        frame = frame.cpu().numpy()
    arr = np.squeeze(np.array(frame))
    if arr.dtype in (np.float32, np.float64, np.float16):
        arr = (arr * 255).clip(0, 255).astype(np.uint8) if arr.max() <= 1.0 else arr.clip(0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def extract_scene_environment(story_text):
    """Extract the PRIMARY environment from the story for all keyframes.
    Returns a consistent scene description used across ALL shots."""
    lower = story_text.lower()

    # Detect primary environment
    env_keywords = {
        "cherry blossom forest": "lush cherry blossom forest, pink sakura petals floating in golden sunset light, ancient trees with twisted trunks, soft volumetric god rays filtering through canopy",
        "forest": "dense mystical forest, dappled sunlight through leaves, ancient trees, ethereal atmosphere",
        "beach": "pristine anime beach, crystal turquoise water, golden sand, dramatic sunset sky",
        "city": "futuristic anime cityscape, neon lights reflecting on wet streets, towering buildings",
        "school": "Japanese high school courtyard, cherry trees, warm afternoon light",
        "temple": "ancient Japanese temple, stone lanterns, moss-covered stairs, misty atmosphere",
        "mountain": "majestic mountain landscape, dramatic clouds, alpine flowers, vast sky",
        "castle": "grand fantasy castle, towering spires, magical aurora in sky",
        "village": "quaint Japanese village, traditional wooden houses, lantern-lit streets",
        "garden": "serene Japanese garden, koi pond, stone bridge, bamboo fence",
        "bridge": "traditional wooden bridge over a gentle river, cherry blossom petals on water",
        "river": "peaceful riverside, gentle flowing water, wildflowers along banks",
        "night": "moonlit landscape, star-filled sky, soft blue ambient light",
        "sunset": "golden hour landscape, warm orange and pink sky, long dramatic shadows",
        "rain": "rain-soaked streets, puddle reflections, moody grey-blue atmosphere",
    }

    # Find the FIRST matching environment — this becomes the consistent scene
    matched_env = None
    for keyword, desc in env_keywords.items():
        if keyword in lower:
            matched_env = desc
            break

    if not matched_env:
        matched_env = "detailed anime environment, atmospheric lighting, painterly background"

    # Add time-of-day if mentioned
    if "sunset" in lower or "evening" in lower:
        matched_env += ", golden hour sunset lighting, warm orange ambient"
    elif "night" in lower or "moon" in lower:
        matched_env += ", moonlit atmosphere, cool blue tones"
    elif "morning" in lower or "dawn" in lower:
        matched_env += ", soft morning light, gentle mist"

    return matched_env

def build_continuous_shots(story_text, character_name):
    """Decompose story into shots that share ONE continuous scene.
    Each shot varies only in CHARACTER ACTION, not environment."""
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', story_text.strip()) if s.strip()]

    # Extract the single consistent environment
    environment = extract_scene_environment(story_text)

    shots = []
    # Camera progression for cinematic feel — one continuous sequence
    camera_sequence = [
        "wide establishing shot slowly pushing in",
        "medium shot tracking alongside, gentle dolly",
        "close-up portrait shot, shallow depth of field",
        "medium-wide pullback shot, scenic composition",
        "over-the-shoulder shot, atmospheric depth",
    ]

    for i, sentence in enumerate(sentences):
        # Extract the ACTION from the sentence (what character does)
        action = sentence.strip().rstrip(".")

        camera = camera_sequence[i % len(camera_sequence)]

        shots.append({
            "description": f"{environment}",
            "action": action,
            "camera": camera,
            "characters": [character_name] if character_name else ["Character"],
        })

    return shots, environment


# Studio anime style constants
STUDIO_STYLE = (
    "studio anime, makoto shinkai style, ufotable quality, "
    "cel shading with soft gradients, detailed lineart, "
    "cinematic composition, film grain, depth of field, "
    "volumetric lighting, painterly background art, "
    "4K anime wallpaper quality"
)

STUDIO_NEGATIVE = (
    "3d render, cgi, photorealistic, live action, photograph, "
    "blurry, low quality, worst quality, deformed face, ugly, "
    "disfigured, bad anatomy, bad hands, extra fingers, "
    "static image, no motion, watermark, text, logo, "
    "chibi, super deformed, sketch, rough draft, "
    "multiple views, split screen, collage, "
    "western cartoon, disney style"
)

# ================================================================
# Phase 1: Story Decomposition (continuous scene)
# ================================================================
print("=" * 60)
print("PHASE 1: Story Decomposition (single continuous scene)")
print("=" * 60)

shots, scene_env = build_continuous_shots(STORY_TEXT, CHARACTER_NAME)

print(f"\nScene environment (shared across all shots):")
print(f"  {scene_env[:100]}...")
print(f"\nGenerated {len(shots)} shots (same background, different actions):")
for i, s in enumerate(shots):
    print(f"  Shot {i+1}: [{s['camera'][:30]}] {s['action'][:60]}...")

# ================================================================
# Phase 2: SDXL + LoRA Keyframes (consistent scene)
# ================================================================
print("\n" + "=" * 60)
print("PHASE 2: SDXL + LoRA Keyframes (consistent scene)")
print("=" * 60)

gc.collect(); torch.cuda.empty_cache()

char_id = CHARACTER_NAME.lower().replace(" ", "_") if CHARACTER_NAME else None
lora_dir = WAREHOUSE / "lora" / char_id if char_id else None
has_lora = lora_dir and (lora_dir / "metadata.json").exists() if lora_dir else False

from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

if has_lora:
    from peft import PeftModel
    meta = json.loads((lora_dir / "metadata.json").read_text())
    base_model = meta.get("base_model", "cagliostrolab/animagine-xl-3.1")
    if base_model.startswith("/") or base_model.startswith("C:"):
        base_model = "cagliostrolab/animagine-xl-3.1"
    print(f"Loading {base_model} + LoRA for '{CHARACTER_NAME}'...")
else:
    base_model = "cagliostrolab/animagine-xl-3.1"
    print(f"Loading {base_model} (no character LoRA)...")

pipe = StableDiffusionXLPipeline.from_pretrained(
    base_model, torch_dtype=torch.float16,
    cache_dir=str(WAREHOUSE / "models"),
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

if has_lora and (lora_dir / "adapter_model.safetensors").exists():
    pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
    pipe.unet.eval()
    if LORA_SCALE != 1.0:
        for module in pipe.unet.modules():
            if hasattr(module, "scaling"):
                for key in module.scaling:
                    module.scaling[key] = LORA_SCALE
    print(f"  LoRA loaded (scale={LORA_SCALE})")

pipe.to("cuda")
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
try: pipe.enable_xformers_memory_efficient_attention()
except: pass

keyframes = []
for i, shot in enumerate(shots):
    # ALL keyframes share the SAME environment description
    # Only the character pose/action changes slightly
    char_tag = CHARACTER_NAME.replace("_", " ").title() if CHARACTER_NAME else "anime girl"

    # Keyframe prompt: consistent scene + face detail + studio quality
    prompt = (
        f"1girl, {char_tag}, "
        f"beautiful detailed face, clear sharp eyes, expressive anime eyes, "
        f"detailed hair, "
        f"{scene_env}, "
        f"{shot['camera']}, "
        f"{STUDIO_STYLE}, "
        f"masterpiece, best quality, absurdres"
    )

    gen = torch.Generator("cuda").manual_seed(42 + i)
    result = pipe(
        prompt=prompt, negative_prompt=NEGATIVE_PROMPT,
        height=IMAGE_HEIGHT, width=IMAGE_WIDTH,
        num_inference_steps=SDXL_STEPS, guidance_scale=SDXL_GUIDANCE, generator=gen,
    )
    keyframes.append(result.images[0])
    print(f"  Keyframe {i+1}/{len(shots)}: {shot['camera'][:40]}")

kf_dir = WAREHOUSE / "outputs" / "keyframes"
kf_dir.mkdir(parents=True, exist_ok=True)
for i, kf in enumerate(keyframes):
    kf.save(kf_dir / f"kf_{i:03d}.png")
print(f"Keyframes saved to {kf_dir}")

# Unload SDXL
if has_lora:
    while hasattr(pipe.unet, "base_model"):
        try: pipe.unet = pipe.unet.base_model.model
        except: break
pipe.to("cpu")
del pipe; gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print(f"SDXL unloaded. VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f}GB")

# ================================================================
# Phase 3: Wan2.1 Image-to-Video
# ================================================================
print("\n" + "=" * 60)
print("PHASE 3: Wan2.1 Image-to-Video")
print("=" * 60)
print("Keyframe → animated video (gentle motion, face preserved)\n")

all_clips = []
start_time = time.time()

from diffusers import WanImageToVideoPipeline

WAN_MODEL = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"
print(f"Loading {WAN_MODEL}...")

wan_pipe = WanImageToVideoPipeline.from_pretrained(
    WAN_MODEL, torch_dtype=torch.float16,
    cache_dir=str(WAREHOUSE / "models"),
)
if OFFLOAD_MODE == "model":
    wan_pipe.enable_model_cpu_offload()
else:
    wan_pipe.enable_sequential_cpu_offload()
wan_pipe.vae.enable_slicing()
wan_pipe.vae.enable_tiling()
print(f"Wan2.1 I2V ready — {OFFLOAD_MODE} offload, {WAN_W}x{WAN_H}\n")

# Motion descriptions — GENTLE and SUBTLE to prevent face/body distortion
# Key: tell Wan WHAT MOVES (hair, petals, clothes) not the character's body
wan_motion_cues = [
    "gentle wind blowing hair and petals, character walking slowly forward, subtle body sway",
    "soft breeze moving hair, character standing still looking around, leaves floating gently",
    "hair flowing in wind, character turns head slightly, cherry petals drifting down slowly",
]

for i, kf_image in enumerate(keyframes):
    shot = shots[i]
    motion = wan_motion_cues[i % len(wan_motion_cues)]

    # Wan prompt: focus on WHAT MOVES, keep face/body stable
    wan_prompt = (
        f"{motion}, "
        f"anime girl in {scene_env[:60]}, "
        f"maintaining character appearance, same face throughout, "
        f"smooth slow gentle animation, subtle motion, "
        f"studio anime quality, cinematic, film grain"
    )
    print(f"  Shot {i+1}/{len(keyframes)}: {motion[:55]}...")
    print(f"    VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f}GB")

    kf_resized = kf_image.resize((WAN_W, WAN_H), Image.LANCZOS)
    gen = torch.Generator("cpu").manual_seed(42 + i)

    result = wan_pipe(
        image=kf_resized,
        prompt=wan_prompt,
        negative_prompt=STUDIO_NEGATIVE,
        num_frames=NUM_FRAMES,
        width=WAN_W,
        height=WAN_H,
        num_inference_steps=WAN_STEPS,
        guidance_scale=WAN_GUIDANCE,
        generator=gen,
    )
    clip_frames = result.frames[0]
    processed_frames = [wan_frame_to_pil(f) for f in clip_frames]

    if len(processed_frames) >= 2:
        f0 = np.array(processed_frames[0]).astype(np.float32)
        f_last = np.array(processed_frames[-1]).astype(np.float32)
        diff = np.mean(np.abs(f0 - f_last))
        print(f"    {'Motion detected' if diff >= 1.0 else 'WARNING: nearly static'} (diff={diff:.1f})")

    all_clips.append(processed_frames)
    print(f"    {len(processed_frames)} frames")
    gc.collect(); torch.cuda.empty_cache()

anim_time = time.time() - start_time
print(f"\nWan2.1 I2V done in {anim_time/60:.1f} min")
del wan_pipe; gc.collect(); torch.cuda.empty_cache()

# ================================================================
# Phase 4: Post-Processing
# ================================================================
print("\n" + "=" * 60)
print("PHASE 4: Post-Processing")
print("=" * 60)

# 4a. Trim static tails
try:
    from agents.postprocess.motion_trim import MotionTrimmer
    trimmer = MotionTrimmer()
    for idx, clip in enumerate(all_clips):
        orig = len(clip)
        all_clips[idx] = trimmer.trim_static_tail(clip)
        trimmed = orig - len(all_clips[idx])
        if trimmed > 0:
            print(f"  Clip {idx+1}: trimmed {trimmed} static frames")
except Exception as e:
    print(f"  Motion trim skipped: {e}")

# 4b. Face restoration
if FACE_RESTORE:
    try:
        from agents.postprocess.face_restore import AnimeFaceRestorer
        restorer = AnimeFaceRestorer()
        print("Restoring anime faces...")
        for idx, clip in enumerate(all_clips):
            all_clips[idx] = restorer.restore_frames(clip)
        print("  Done")
    except Exception as e:
        print(f"  Face restore skipped: {e}")

# 4c. Spatial upscale / sharpening
if SPATIAL_UPSCALE:
    try:
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan import RealESRGANer
        esrgan_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)
        esrgan = RealESRGANer(
            scale=4,
            model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",
            model=esrgan_model, half=True,
        )
        print("Sharpening with Real-ESRGAN...")
        for idx, clip in enumerate(all_clips):
            sharpened = []
            for frame_pil in clip:
                frame_bgr = cv2.cvtColor(np.array(frame_pil), cv2.COLOR_RGB2BGR)
                h_orig, w_orig = frame_bgr.shape[:2]
                out_bgr, _ = esrgan.enhance(frame_bgr, outscale=1)
                if out_bgr.shape[:2] != (h_orig, w_orig):
                    out_bgr = cv2.resize(out_bgr, (w_orig, h_orig), interpolation=cv2.INTER_LANCZOS4)
                sharpened.append(Image.fromarray(cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)))
            all_clips[idx] = sharpened
        del esrgan, esrgan_model; gc.collect(); torch.cuda.empty_cache()
        print("  Done")
    except Exception as e:
        print(f"  Real-ESRGAN skipped: {e}")

# 4d. Color grading
if COLOR_GRADE:
    try:
        from agents.postprocess.color_grade import AnimeColorGrader
        grader = AnimeColorGrader()
        print(f"Color grading ({COLOR_PALETTE})...")
        for idx, clip in enumerate(all_clips):
            all_clips[idx] = grader.grade_with_palette(clip, palette=COLOR_PALETTE)
        print("  Done")
    except Exception as e:
        print(f"  Color grading skipped: {e}")

# ================================================================
# Phase 5: Temporal Upscale (RIFE)
# ================================================================
if TARGET_FPS > FPS:
    print(f"\nTemporal upscale: {FPS}fps -> {TARGET_FPS}fps")
    multiplier = max(2, round(TARGET_FPS / FPS))
    rife_ok = False
    try:
        from agents.postprocess.upscaler import VideoUpscaler
        _up = VideoUpscaler(str(WAREHOUSE))
        _rife = _up._load_rife()
        if _rife: rife_ok = True; print("  Using RIFE")
    except: pass

    for idx, clip in enumerate(all_clips):
        orig = len(clip)
        if rife_ok:
            try:
                all_clips[idx] = _up._rife_interpolate_sequence(_rife, clip, multiplier)
            except:
                rife_ok = False
        if not rife_ok:
            upscaled = []
            for j in range(len(clip) - 1):
                upscaled.append(clip[j])
                a1 = np.array(clip[j]).astype(np.float32)
                a2 = np.array(clip[j+1]).astype(np.float32)
                for k in range(1, multiplier):
                    alpha = k / multiplier
                    upscaled.append(Image.fromarray(((1-alpha)*a1 + alpha*a2).astype(np.uint8)))
            upscaled.append(clip[-1])
            all_clips[idx] = upscaled
        print(f"  Clip {idx+1}: {orig} -> {len(all_clips[idx])} frames")

    if rife_ok:
        try: _up._unload_rife()
        except: pass
        gc.collect(); torch.cuda.empty_cache()
    output_fps = TARGET_FPS
else:
    output_fps = FPS

# ================================================================
# Phase 6: Assembly — longer cross-dissolve for smooth scene flow
# ================================================================
print(f"\nAssembling {len(all_clips)} clips (long cross-dissolve)...")
CROSSFADE = 12  # Longer dissolve for seamless continuous feel
all_frames = []
for idx, clip in enumerate(all_clips):
    arrays = [np.array(f) for f in clip]
    if idx == 0:
        all_frames.extend(arrays)
    else:
        overlap = min(CROSSFADE, len(all_frames), len(arrays))
        for j in range(overlap):
            t = (j + 1) / (overlap + 1)
            prev = all_frames[-(overlap - j)].astype(np.float32)
            nxt = arrays[j].astype(np.float32)
            all_frames[-(overlap - j)] = ((1-t)*prev + t*nxt).clip(0, 255).astype(np.uint8)
        all_frames.extend(arrays[overlap:])

duration = len(all_frames) / output_fps
print(f"Total: {len(all_frames)} frames ({duration:.1f}s at {output_fps}fps)")

output_dir = WAREHOUSE / "outputs" / "text2anime"
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / f"anime_{int(time.time())}.mp4"

h, w = all_frames[0].shape[:2]
writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), output_fps, (w, h))
for frame in all_frames:
    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
writer.release()

total_time = time.time() - start_time
size_mb = video_path.stat().st_size / 1e6

print(f"\n{'=' * 60}")
print(f"DONE!")
print(f"{'=' * 60}")
print(f"Video: {video_path}")
print(f"Duration: {duration:.1f}s | Resolution: {w}x{h} | FPS: {output_fps}")
print(f"Size: {size_mb:.1f}MB | Time: {total_time/60:.1f} min")

import matplotlib.pyplot as plt
cols = min(len(keyframes), 4)
rows = (len(keyframes) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
axes_flat = np.array(axes).flat if hasattr(axes, 'flat') else [axes]
for i, (ax, kf) in enumerate(zip(axes_flat, keyframes)):
    ax.imshow(kf)
    ax.set_title(f"Shot {i+1}: {shots[i]['action'][:35]}...", fontsize=9)
    ax.axis("off")
for j in range(i+1, rows*cols):
    try: axes_flat[j].axis("off")
    except: pass
plt.suptitle("Keyframes (same scene, different actions)", fontsize=13)
plt.tight_layout()
plt.show()

gc.collect(); torch.cuda.empty_cache()
print("VRAM freed.")

In [ ]:
#@title 4. Play Video
#@markdown Displays the generated video inline.

from IPython.display import Video, display
from pathlib import Path
import glob

# Find latest video
output_dir = Path("/workspace/warehouse/outputs/text2anime")
videos = sorted(output_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
if videos:
    latest = videos[0]
    print(f"Playing: {latest}")
    print(f"Size: {latest.stat().st_size / 1e6:.1f}MB")
    display(Video(str(latest), embed=True, width=512))
else:
    print("No videos found. Run Cell 3 first.")

In [ ]:
#@title 5. Download Video
#@markdown Downloads the latest video to your local machine.

from pathlib import Path
from IPython.display import FileLink

output_dir = Path("/workspace/warehouse/outputs/text2anime")
videos = sorted(output_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
if videos:
    print(f"Right-click to download: {videos[0].name}")
    display(FileLink(str(videos[0])))
else:
    print("No videos found.")

In [ ]:
#@title 6. Stop Pod (saves money!)
#@markdown Stops the pod to avoid charges. Your /workspace volume persists.

STOP_POD = False  #@param {type:"boolean"}

if STOP_POD:
    import subprocess
    subprocess.run(["runpodctl", "stop", "pod", "$RUNPOD_POD_ID"], check=False)
    print("Pod stopping... Your volume data is preserved.")
else:
    print("Toggle STOP_POD to True and re-run to stop.")